In [1]:
import torch

print("PyTorch:", torch.__version__)

PyTorch: 2.10.0


# 在 CPU 和 MPS 上实现可复现的 Token 采样

- 当我们使用大语言模型生成文本时，通常会使用 `torch.softmax` 将 logits 转换为概率分布，并使用 `torch.multinomial` 对下一个 token 进行采样。
- 不幸的是，即使使用了相同的随机种子，CPU 和 MPS 有时也会从完全相同的概率分布中采样出不同的 token。这很可能是因为每种设备都使用自己独立的随机数生成器。
- 在本笔记本中，我用一个简单的示例来展示这种现象，并通过在两种情况下都在 CPU 上运行采样步骤，使采样结果保持一致。
- 顺便提一下，PyTorch 并不保证跨设备或跨版本之间能产生完全相同的结果。详情请参阅其[可复现性说明](https://docs.pytorch.org/docs/stable/notes/randomness.html)。
- 本文的示例刻意保持极简。有关包含温度缩放（temperature scaling）的完整大语言模型文本生成函数，请参阅[第 5 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/ch05.ipynb)。

&nbsp;
## 1) 在各设备上分别进行采样

- 以下代码在每次调用前将随机种子硬编码设置为 `123`，以便我们在相同的种子下对比 CPU 和 MPS 的表现。

In [2]:
logits = torch.tensor(
    [[0.0, 1.0, 2.0]],
    dtype=torch.bfloat16,
)

# Convert logits to probabilities
probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

def sample(probs):

    torch.manual_seed(123)

    # Sample from the distribution
    idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

    return idx_next


print("CPU:", sample(probs.to("cpu")))

if torch.mps.is_available():
    print("MPS:", sample(probs.to("mps")))

if torch.cuda.is_available():
    print("CUDA:", sample(probs.to("cuda")))

CPU: tensor([[1]])
MPS: tensor([[2]], device='mps:0')


&nbsp;
## 2) 两种设备均在 CPU 上进行采样

- 因此，针对上述 CPU/MPS 不一致问题的一种修复方案或变通方法，是在调用 `torch.multinomial` 之前先执行 `probs.cpu()`。

In [3]:
def sample_on_cpu(probs):

    torch.manual_seed(123)

    # Sample from the distribution
    idx_next = torch.multinomial(probs.cpu(), num_samples=1)  # (batch_size, 1)

    return idx_next.to(probs.device)


print("CPU:", sample_on_cpu(probs.to("cpu")))

if torch.mps.is_available():
    print("MPS:", sample_on_cpu(probs.to("mps")))

if torch.cuda.is_available():
    print("CUDA:", sample_on_cpu(probs.to("cuda")))

CPU: tensor([[1]])
MPS: tensor([[1]], device='mps:0')


- 顺便提一下，这只是 MPS 和 CPU 产生差异的原因之一。
- 还有其他可能的原因，例如在完整的 LLM 中，CPU 和 MPS 可能会计算出略有差异的 logits。这些细微差异会改变概率分布和采样的 token，因此即使在 CPU 上运行采样步骤，生成的文本可能仍会有所不同。
- 此外，某些问题是由部分芯片（如 M1 和 M2）中的硬件/驱动缺陷引起的，正如 Scott Kirila 在精彩的调查文章《[在闭源着色器内核中寻找 Bug](https://scottkirila.studio.site/blog/kernel-bug)》（Finding a Bug in Closed-Source Shader Kernels）中所描述的那样。